In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

num_workers = 2 if torch.cuda.is_available() else 0
pin_memory = torch.cuda.is_available()


In [ ]:
# Domain investigation (model domain vs dataset domain)
# Model domain: supervised image classification where the output is a discrete class label.
# Dataset domain: Oxford-IIIT Pet images (cats/dogs) with breed-like categories.

# Golden rule (from your screenshot): augmentations must preserve the label.
# So we do safe changes like small rotations, horizontal flips, and cropping,
# but we avoid label-breaking transforms (big rotations/vertical flips).

print('Goal: train MLP -> CNN -> transfer learning (3-A/3-B/3-C), show loss + val metrics, and visualize filters.')


In [ ]:
# Data: use torchvision.datasets directly (Colab-friendly)
# This removes the need for tmp_data entirely.

data_dir = './oxford_iiit_pet_data'

# Load without transforms first, just to read targets for a stratified split.
ds_trainval_noaug = datasets.OxfordIIITPet(
    root=data_dir,
    split='trainval',
    target_types='category',
    download=True,
    transform=None
)

class_names = getattr(ds_trainval_noaug, 'classes', None)
if class_names is None:
    # Fallback if torchvision exposes classes differently
    class_names = [str(i) for i in range(0, 40)]

def get_targets_from_dataset(ds):
    # Try common public attributes first (fast), otherwise fallback to iterating.
    for attr in ['targets', 'labels', 'y']:
        if hasattr(ds, attr):
            t = getattr(ds, attr)
            return np.asarray(t, dtype=np.int64)
    ys = []
    for i in range(len(ds)):
        _, y = ds[i]
        if isinstance(y, (tuple, list)):
            y = y[0]
        ys.append(int(y))
    return np.asarray(ys, dtype=np.int64)

targets = get_targets_from_dataset(ds_trainval_noaug)
num_classes = int(targets.max()) + 1
print('num_classes:', num_classes)
print('dataset domain comment: pet breeds/categories, so label identity should survive safe crops/flips.')

def stratified_split_indices(targets_np, val_ratio=0.2, seed=42):
    rng = np.random.default_rng(seed)
    targets_np = np.asarray(targets_np)
    train_indices = []
    val_indices = []
    for c in range(num_classes):
        idx_c = np.where(targets_np == c)[0]
        rng.shuffle(idx_c)
        n_val = int(len(idx_c) * val_ratio)
        val_indices.extend(idx_c[:n_val].tolist())
        train_indices.extend(idx_c[n_val:].tolist())
    rng.shuffle(train_indices)
    rng.shuffle(val_indices)
    return train_indices, val_indices

val_ratio = 0.2
train_indices, val_indices = stratified_split_indices(targets, val_ratio=val_ratio, seed=SEED)
print('train size:', len(train_indices), 'val size:', len(val_indices))

# Transforms (MLP/CNN -> 64x64)
mean_scratch = (0.5, 0.5, 0.5)
std_scratch = (0.5, 0.5, 0.5)

train_tf_64 = transforms.Compose([
    transforms.Resize((72, 72)),
    transforms.RandomResizedCrop(64, scale=(0.75, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean_scratch, std_scratch),
])

val_tf_64 = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean_scratch, std_scratch),
])

# Transforms (ResNet18 -> 224x224)
resnet_mean = (0.485, 0.456, 0.406)
resnet_std = (0.229, 0.224, 0.225)

train_tf_224 = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(resnet_mean, resnet_std),
])

val_tf_224 = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(resnet_mean, resnet_std),
])

# Datasets with transforms (same split indices)
ds_64_train = datasets.OxfordIIITPet(
    root=data_dir,
    split='trainval',
    target_types='category',
    download=False,
    transform=train_tf_64
)
ds_64_val = datasets.OxfordIIITPet(
    root=data_dir,
    split='trainval',
    target_types='category',
    download=False,
    transform=val_tf_64
)

ds_224_train = datasets.OxfordIIITPet(
    root=data_dir,
    split='trainval',
    target_types='category',
    download=False,
    transform=train_tf_224
)
ds_224_val = datasets.OxfordIIITPet(
    root=data_dir,
    split='trainval',
    target_types='category',
    download=False,
    transform=val_tf_224
)

train_ds_64 = Subset(ds_64_train, train_indices)
val_ds_64 = Subset(ds_64_val, val_indices)

train_ds_224 = Subset(ds_224_train, train_indices)
val_ds_224 = Subset(ds_224_val, val_indices)

batch_size_64 = 32
batch_size_224 = 16

train_loader_64 = DataLoader(train_ds_64, batch_size=batch_size_64, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
val_loader_64 = DataLoader(val_ds_64, batch_size=batch_size_64, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

train_loader_224 = DataLoader(train_ds_224, batch_size=batch_size_224, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
val_loader_224 = DataLoader(val_ds_224, batch_size=batch_size_224, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

print('Loaders ready: 64x64 for MLP/CNN and 224x224 for transfer learning.')

# Lazy-loading comment: DataLoader loads images per batch, so we do not move the full dataset into GPU memory.
# True lazy loading (reading from disk on demand inside Dataset) would come next if needed.


In [ ]:
# Sanity visualization: random images with labels
ds_vis = datasets.OxfordIIITPet(
    root=data_dir,
    split='trainval',
    target_types='category',
    download=False,
    transform=transforms.ToTensor()
)

idxs = np.random.choice(len(ds_vis), size=8, replace=False)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()
for ax, i in zip(axes, idxs):
    img, y = ds_vis[i]
    if hasattr(ds_vis, 'classes'):
        name = ds_vis.classes[int(y)]
    else:
        name = str(int(y))
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(name, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Metrics: accuracy + macro recall
def accuracy_and_macro_recall(logits, targets, num_classes):
    preds = logits.argmax(dim=1)
    correct = (preds == targets).sum().item()
    total = targets.numel()
    acc = correct / max(total, 1)

    confmat = torch.zeros(num_classes, num_classes, device=logits.device, dtype=torch.int64)
    confmat.index_put_((targets, preds), torch.ones_like(targets, dtype=torch.int64), accumulate=True)

    tp = torch.diag(confmat).to(torch.float32)
    fn = confmat.sum(dim=1).to(torch.float32) - tp
    recall_per_class = tp / (tp + fn + 1e-8)
    macro_recall = recall_per_class.mean().item()
    return acc, macro_recall

def train_one_model(model, train_loader, val_loader, epochs, lr, freeze_description=''):
    model = model.to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.Adam(params, lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = {
        'train_loss': [],
        'val_loss': [],
        'val_acc': [],
        'val_macro_recall': [],
    }

    print('\nTraining:', model.__class__.__name__, '|', freeze_description)
    print('Trainable params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        seen = 0

        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * yb.size(0)
            seen += yb.size(0)

        train_loss = running_loss / max(seen, 1)

        model.eval()
        val_loss_total = 0.0
        val_seen = 0
        all_acc = []
        all_recall = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)

                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss_total += loss.item() * yb.size(0)
                val_seen += yb.size(0)

                acc, macro_recall = accuracy_and_macro_recall(logits, yb, num_classes=num_classes)
                all_acc.append(acc)
                all_recall.append(macro_recall)

        val_loss = val_loss_total / max(val_seen, 1)
        val_acc = float(np.mean(all_acc))
        val_macro_recall = float(np.mean(all_recall))

        history['train_loss'].append(float(train_loss))
        history['val_loss'].append(float(val_loss))
        history['val_acc'].append(float(val_acc))
        history['val_macro_recall'].append(float(val_macro_recall))

        print(f"Epoch {epoch}/{epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | val_macro_recall={val_macro_recall:.4f}")

    return model, history

def plot_history(history, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(history['train_loss'], label='train_loss')
    ax[0].plot(history['val_loss'], label='val_loss')
    ax[0].set_title(title + ' loss')
    ax[0].legend()
    ax[1].plot(history['val_acc'], label='val_acc')
    ax[1].plot(history['val_macro_recall'], label='val_macro_recall')
    ax[1].set_title(title + ' metrics')
    ax[1].legend()
    plt.show()


In [ ]:
# 1) Simple MLP trained from scratch
# Comment: MLP treats pixels as a flat vector, so it does not naturally use spatial locality.
class MLP(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        input_dim = 3 * 64 * 64
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.net(x)

epochs_mlp = 5
lr_mlp = 1e-3

mlp = MLP(num_classes=num_classes)
mlp, hist_mlp = train_one_model(mlp, train_loader_64, val_loader_64, epochs=epochs_mlp, lr=lr_mlp, freeze_description='all layers trainable')

best_idx = int(np.argmax(hist_mlp['val_acc']))
print(f"MLP comment: best val_acc={hist_mlp['val_acc'][best_idx]:.4f}, best macro_recall={hist_mlp['val_macro_recall'][best_idx]:.4f}. It may lag CNNs because it ignores spatial edges/parts.")
plot_history(hist_mlp, 'MLP')


In [ ]:
# 2) Migrate to CNN
# Comment: CNN filters learn reusable local patterns (edges/textures), which is aligned with image domains.
class SmallCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU(inplace=True)

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        return self.head(x)

def show_conv_filters(conv_weight, title, n_filters=8):
    # conv_weight: [out_channels, in_channels, kH, kW]
    w = conv_weight.detach().cpu().numpy()
    out_channels = w.shape[0]
    idxs = np.random.choice(out_channels, size=min(n_filters, out_channels), replace=False)

    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    axes = axes.flatten()
    for ax, i in zip(axes, idxs):
        # Average across input channels for a single greyscale visualization.
        filt = w[i].mean(axis=0)
        vmin, vmax = filt.min(), filt.max()
        if abs(vmax - vmin) < 1e-8:
            filt_norm = np.zeros_like(filt)
        else:
            filt_norm = (filt - vmin) / (vmax - vmin)
        ax.imshow(filt_norm, cmap='gray')
        ax.set_title(f'filter {i}', fontsize=9)
        ax.axis('off')
    for j in range(len(idxs), len(axes)):
        axes[j].axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

cnn = SmallCNN(num_classes=num_classes)
filters_before = cnn.conv1.weight.clone().detach()
show_conv_filters(filters_before, title='CNN conv1 filters BEFORE training')

epochs_cnn = 8
lr_cnn = 1e-3
cnn, hist_cnn = train_one_model(cnn, train_loader_64, val_loader_64, epochs=epochs_cnn, lr=lr_cnn, freeze_description='all layers trainable (CNN)')

filters_after = cnn.conv1.weight.clone().detach()
show_conv_filters(filters_after, title='CNN conv1 filters AFTER training')

best_idx = int(np.argmax(hist_cnn['val_acc']))
print(f"CNN comment: best val_acc={hist_cnn['val_acc'][best_idx]:.4f}, best macro_recall={hist_cnn['val_macro_recall'][best_idx]:.4f}. If it beats MLP, it means the conv layers learned useful edge/texture patterns.")
plot_history(hist_cnn, 'CNN')


In [ ]:
# 3) Transfer learning with pretrained ResNet18
# Comment: pretrained backbones already encode generic visual features.
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.DEFAULT

def set_trainable(model, mode):
    # mode: 'fc_only' | 'layer4_fc' | 'full'
    if mode == 'fc_only':
        for p in model.parameters():
            p.requires_grad = False
        model.fc.requires_grad_(True)
    elif mode == 'layer4_fc':
        for p in model.parameters():
            p.requires_grad = False
        for p in model.layer4.parameters():
            p.requires_grad = True
        model.fc.requires_grad_(True)
    elif mode == 'full':
        for p in model.parameters():
            p.requires_grad = True
    else:
        raise ValueError('unknown mode')

def make_resnet(num_classes):
    m = resnet18(weights=weights)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

# Scenario 3-A: Just Classification FC Layers
epochs_tl_a = 3
lr_tl_a = 1e-3
model_a = make_resnet(num_classes)
set_trainable(model_a, mode='fc_only')
model_a, hist_a = train_one_model(model_a, train_loader_224, val_loader_224, epochs=epochs_tl_a, lr=lr_tl_a, freeze_description='3-A: freeze backbone, train fc only')
best_idx = int(np.argmax(hist_a['val_acc']))
print(f"Transfer 3-A comment: best val_acc={hist_a['val_acc'][best_idx]:.4f}, best macro_recall={hist_a['val_macro_recall'][best_idx]:.4f}. If it is already good, ImageNet features transfer well.")
plot_history(hist_a, 'Transfer A (FC only)')

# Scenario 3-B: Finetune the final layers (layer4 + fc)
epochs_tl_b = 4
lr_tl_b = 1e-4
model_b = make_resnet(num_classes)
set_trainable(model_b, mode='layer4_fc')
model_b, hist_b = train_one_model(model_b, train_loader_224, val_loader_224, epochs=epochs_tl_b, lr=lr_tl_b, freeze_description='3-B: unfreeze layer4 + fc')
best_idx = int(np.argmax(hist_b['val_acc']))
print(f"Transfer 3-B comment: best val_acc={hist_b['val_acc'][best_idx]:.4f}, best macro_recall={hist_b['val_macro_recall'][best_idx]:.4f}. Unfreezing layer4 helps adapt high-level features to pet breeds.")
plot_history(hist_b, 'Transfer B (layer4+fc)')

# Scenario 3-C: Finetune the whole model
epochs_tl_c = 3
lr_tl_c = 1e-5
model_c = make_resnet(num_classes)
filters_before_resnet = model_c.conv1.weight.clone().detach()
show_conv_filters(filters_before_resnet, title='ResNet conv1 filters BEFORE training (3-C)')
set_trainable(model_c, mode='full')
model_c, hist_c = train_one_model(model_c, train_loader_224, val_loader_224, epochs=epochs_tl_c, lr=lr_tl_c, freeze_description='3-C: unfreeze everything (small lr)')
filters_after_resnet = model_c.conv1.weight.clone().detach()
show_conv_filters(filters_after_resnet, title='ResNet conv1 filters AFTER training (3-C)')

best_idx = int(np.argmax(hist_c['val_acc']))
print(f"Transfer 3-C comment: best val_acc={hist_c['val_acc'][best_idx]:.4f}, best macro_recall={hist_c['val_macro_recall'][best_idx]:.4f}. Full finetune can win, but it may overfit if the dataset is small.")
plot_history(hist_c, 'Transfer C (full finetune)')
